In [8]:
# 05/14 10:00 기준 기능 다 동작하는거 확인

from netmiko import ConnectHandler, BaseConnection

import os
import re
import json
"""
* Cisco 라우터/스위치 세팅 자동화

* SSH 기반- 세팅할 라우터/스위치에는 SSH로 접근 가능한 상태여야 함
 # conf t
 (config)# username 유저명 privilege 권한레벨 password 비밀번호
 (config)# ip domain-name 키_시드값
 (config)# crypto key generate rsa -> 1024 입력
 (config)# ip ssh ver 2 -> SSHv2 사용
 (config)# line vty 0 4 -> 0~4번까지 접속허용(5명)
 (config)# login local  -> local에 설정한 정보 기반으로 로그인 허용
 (config)# transport input ssh -> netmiko는 SSH 기반으로 하자

 자동화된 기능들
  - netmiko SSH 커넥션 얻어오기
  - 커맨드 실행하기
  - 커맨드 configure terminal 에서 실행하기
  - 커넥션 통해 라우터인지 스위치인지 알아내기(c3745는 스위치, c7200은 라우터로 판정)
  - 커넥션 통해 해당 장비의 이름 알아내기(R1, ESW1 같은거)
  - show ip interface brief 정보 딕셔너리 형태로 가공해 알아내기
  - 라우터/스위치의 DHCP 서버화
  - 라우터의 NAT(현재 PAT 방식만 지원) 설정 자동화
  - 라우팅 테이블 가공해 알아내기
  - 라우팅 테이블/DHCP 서버/NAT 설정 프로젝트 단위로 백업/복구하는 기능

 수정이 필요한 부분들
  - NAT 설정 방식이 PAT밖에 지원되지 않음
  - 명령 실행 시 스위치인지 라우터인지 확인하는 루틴이 없음(기능은 작동하지만 에러 발생 가능성 존재)
  - 복구 시 파일에 해당 장비 이름이 없을 경우 오류 처리하는 루틴이 없음
  - 라우팅 테이블의 AD값 세팅 기능은 없음
  + 아직 발견되지 못한 오류들?

"""

# 모든 함수는 con: Baseconnection을 Parameter로 받음
#  - 여기서 con은 세팅할 라우터/스위치와의 SSH 연결 상태여야 함

# Cisco 네트워크 장비의 SSH 연결 객체(netmiko.BaseConnection)을 얻어오는 함수
def get_con(ip: str, 
            username: str="root", 
            password: str="cisco", 
            port: int=22, 
            device_type: str="cisco_ios") -> BaseConnection:
    con = ConnectHandler(**{"device_type": device_type, "host": ip, "username": username, "password": password, "port": port})
    con.enable()
    return con


# 명령어를 일반 상태에서 실행시켜 주는 함수 -> 실행 결과를 str type으로 리턴
def run_cmd(con: BaseConnection, 
            cmd: str) -> str:
    return con.send_command(cmd)


# 명령어를 configure terminal 상태에서 실행시켜주는 함수 -> 실행 결과를 str type으로 리턴
def run_conf(con: BaseConnection, 
             cmd: str) -> str:
    return con.send_config_set(cmd)

#명령어를 (vlan) 모드에서 실행시켜주는 함수
def run_vlan(con: BaseConnection,  
             cmds: str) -> str:

    output = ""
    
    #vlan database 모드 진입
    output += con.send_command("vlan database", expect_string=r"vlan\)")
    
    #cmd
    for c in cmds:
        output += con.send_command(c, expect_string=r"vlan\)")
        
    # 설정을 실제로 반영하고 빠져나오기 
    output += con.send_command("exit", expect_string=r"#")
    
    return output

    
def get_device_role(con: BaseConnection) -> str:
    """
    장비의 모델을 확인하여 SWITCH 또는 ROUTER 역할을 반환함
    """
    version_out = run_cmd(con, "show version")
    inventory_out = run_cmd(con, "show inventory")
    
    if "3745" in version_out:
        # 3745이면서 EtherSwitch 모듈이 장착된 경우 SWITCH로 판별
        if "ESW" in inventory_out or "EtherSwitch" in inventory_out:
            return "SWITCH"
        return "ROUTER"
    elif "7200" in version_out:
        return "ROUTER"
    
    return "UNKNOWN"


# 호스트의 이름 리턴해주는 함수
def get_host_name(con: BaseConnection) -> str:
    return con.base_prompt


# ip/프리픽스 형식 입력 시 자동으로 와일드마스크 변환
# 내부에서 사용하기 위한 목적
def _wildcard_(ip_prefix: str) -> str:
    ip, prefix = ip_prefix.split("/")
    prefix = int(prefix)
    # 서브넷 마스크 생성
    subnet = (0xffffffff << (32 - prefix)) & 0xffffffff
    # 옥텟 분리
    mask = [
        (subnet >> 24) & 255,
        (subnet >> 16) & 255,
        (subnet >> 8) & 255,
        subnet & 255
    ]
    # 와일드마스크 생성
    wildcard_mask = [255 - x for x in mask]
    return f"{ip} {'.'.join(map(str, wildcard_mask))}"


# sh ip int bri 딕셔너리로 깔끔하게 가져오자
def get_interface_brief(con: BaseConnection) -> dict:
    txt = run_cmd(con, "show ip interface brief")
    lines = txt.strip().splitlines()
    interface_dict = {}
    for line in lines:
        if line.startswith("Interface") or not line.strip():
            continue
        
        parts = line.split()
        
        if len(parts) >= 6:
            int_name = parts[0]   # 인터페이스 이름
            ip_addr = parts[1]    # 할당된 아이피
            ok_status = parts[2]  # YES?
            method = parts[3]     # 연결상태
            protocol = parts[-1]  # 마지막 요소
            status = " ".join(parts[4:-1]) 

            interface_dict[int_name] = {
                "ip_address": ip_addr,
                "ok": ok_status,
                "method": method,
                "status": status,
                "protocol": protocol
            }
            
    return interface_dict


# 라우터를 DHCP 서버로 설정해 주는 함수 -> 실행 결과를 str type으로 리턴
def router_dhcp_config(con: BaseConnection, 
                       pool_name: str, 
                       nid: str, 
                       subnet_mask: str, 
                       gateway_ip: str, 
                       exclude_ip_start: str, 
                       exclude_ip_end: str, 
                       dns_server_ip: str="8.8.8.8") -> str:
    cmds=[
        f"ip dhcp excluded-address {exclude_ip_start} {exclude_ip_end}",
        f"ip dhcp pool {pool_name}",
        f"network {nid} {subnet_mask}",
        f"dns-server {dns_server_ip}",
        f"default-router {gateway_ip}"
    ]
    return run_conf(con, cmds)


# 스위치를 DHCP 서버로 설정해 주는 함수 -> 실행 결과를 str type으로 리턴
def switch_dhcp_config(con: BaseConnection,
                       pool_name: str, 
                       nid: str, 
                       subnet_mask: str, 
                       gateway_ip: str, 
                       exclude_ip_start: str, 
                       exclude_ip_end: str, 
                       dns_server_ip: str="8.8.8.8") -> str:
    cmds=[
        f"ip dhcp excluded-address {exclude_ip_start} {exclude_ip_end}",
        f"service dhcp", # 스위치이므로 dhcp service를 활성화 시켜주는 부분 추가
        f"ip dhcp pool {pool_name}",
        f"network {nid} {subnet_mask}",
        f"dns-server {dns_server_ip}",
        f"default-router {gateway_ip}"
    ]
    return run_conf(con, cmds)


# 라우터에 NAT(PAT 방식) 설정을 해주는 함수 -> 실행 결과를 str type으로 리턴
def router_pat_nat_config(con: BaseConnection,
                      in_interface: str,
                      out_interface: str,
                      ip_and_netmask_prefix: str) -> str:
                      #nat_ip_pool_name: str,
                      #nat_ip_pool_start_ip: str,
                      #nat_ip_pool_end_ip: str,
                      #nat_ip_pool_netmask: str):
    cmds = [
        f"interface {in_interface}",
        "ip nat inside",
        f"interface {out_interface}",
        "ip nat outside",
        #f"ip nat pool {nat_ip_pool_name} {nat_ip_pool_start_ip} {nat_ip_pool_end_ip} netmask {nat_ip_pool_netmask}",
        f"access-list 1 permit {_wildcard_(ip_and_netmask_prefix)}",
        f"ip nat inside source list 1 int {out_interface} overload"
    ]
    return run_conf(con, cmds)


# 라우터의 현재 라우팅 테이블을 리스트 형태로(원소는 딕셔너리) 얻어오는 함수
def get_routing_table(con: BaseConnection) -> list:
    # 1. 라우팅 테이블 문자열 형태로 얻어오기(sh run)
    table_raw = run_cmd(con, "show run | include ^ip route")
    # 2. 라우팅 테이블 가공하기
    route_list = []
    pattern = r"ip route (\S+) (\S+) (\S+)"
    matches = re.findall(pattern, table_raw)
    
    for m in matches:
        route_list.append({ "NID": m[0], "Netmask": m[1], "Gateway": m[2] })
    
    return route_list


# 라우터에 라우팅 경로 하나 추가하는 함수 -> 결과 문자열로 리턴
def add_rotue_info(con: BaseConnection,
                nid: str,
                netmask: str,
                gateway: str) -> str:
    return run_conf(con, f"ip route {nid} {netmask} {gateway}")

# ========================= 이하 json 관련 함수 =============================

# 특정 라우터의 라우팅 테이블 정보를, 프로젝트 라우팅 정보 json에 저장하는 함수
# !주의! 이름이 같은 기존 라우터의 라우팅 테이블 정보가 이미 프로젝트 라우팅 정보 json에 존재한다면 덮어쓰기됨
# 저장한 라우팅 테이블 정보를 리스트 형태로 반환
def store_routing_table(con: BaseConnection, 
                        routing_table_json_path: str) -> list:
    # 1. 현재 라우터의 라우팅 테이블 정보 가져오기
    routing_table = get_routing_table(con)
    
    # 2. 기본 데이터 구조 초기화
    data = {}

    # 3. 파일이 존재하고 내용이 있는지 확인 후 읽기
    # 파일이 없거나 비어있으면(0 byte) 읽기 과정을 건너뜁니다.
    if os.path.exists(routing_table_json_path) and os.path.getsize(routing_table_json_path) > 0:
        with open(routing_table_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)

    # 4. 데이터 업데이트 (새로운 라우터 정보를 추가하거나 기존 정보를 덮어씀)
    data[get_host_name(con)] = routing_table

    # 5. 파일 저장
    # 'w' 모드는 파일이 없으면 자동으로 생성하고, 있으면 덮어씁니다.
    with open(routing_table_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    return routing_table


# 특정 라우터의 라우팅 테이블 정보 복구하기 -> 특정 라우터만 복구하더라도 전체 프로젝트의 라우팅 테이블 정보 json 필요함
# 복구된 라우팅 테이블 리스트 형태로 반환 
# 실행 시간 때문에 add_route_info 사용하지 않고 구현한 것 유의
def restore_routing_table(con: BaseConnection,
                          routing_table_json_path: str) -> list:
    data = None
    with open(routing_table_json_path, 'r', encoding="utf-8") as f:
        data = json.load(f)
        cmd = []
        for info in data[get_host_name(con)]:
            cmd.append(f"ip route {info['NID']} {info['Netmask']} {info['Gateway']}")
        if len(cmd) > 0:
            run_conf(con, cmd)
    return data[get_host_name(con)]


# DHCP 정보를 가져오는 함수
def get_dhcp_info(con: BaseConnection) -> list[dict]:
    output = run_cmd(con, "show run | sec dhcp").splitlines()
    
    all_pools = []
    current_pool = {}
    excluded_list = [] # 제외 주소들을 모두 담을 리스트

    for line in output:
        line = line.strip()
        
        # 1. 제외 주소 수집 (모든 라인을 리스트에 추가)
        if "excluded-address" in line:
            parts = line.split()
            if len(parts) >= 4:
                start = parts[3]
                end = parts[4] if len(parts) >= 5 else parts[3]
                excluded_list.append({"start": start, "end": end})
            continue

        # 2. 풀 시작
        if line.startswith("ip dhcp pool"):
            if current_pool:
                all_pools.append(current_pool)
            
            current_pool = {
                "Poolname": line.split()[-1],
                "NID": None, "Netmask": None, "Gateway": None,
                "Excluded_Addresses": list(excluded_list), # 지금까지 수집된 리스트 복사
                "DNS_Server_IP": None
            }

        # 3. 상세 정보 (get 메서드 대신 직접 할당 시 current_pool 존재 여부 체크는 필수)
        elif line.startswith("network") and current_pool:
            parts = line.split()
            if len(parts) >= 3:
                current_pool["NID"] = parts[1]
                current_pool["Netmask"] = parts[2]
        elif line.startswith("default-router") and current_pool:
            current_pool["Gateway"] = line.split()[-1]
        elif line.startswith("dns-server") and current_pool:
            current_pool["DNS_Server_IP"] = line.split()[-1]

    if current_pool:
        all_pools.append(current_pool)

    return all_pools


# DHCP 정보를 파일로 백업하는 함수
def store_dhcp_setting(con: BaseConnection, 
                        dhcp_setting_json_path: str) -> list:
    dhcp_info = get_dhcp_info(con)
    
    data = {}
    if os.path.exists(dhcp_setting_json_path) and os.path.getsize(dhcp_setting_json_path) > 0:
        with open(dhcp_setting_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)

    data[get_host_name(con)] = dhcp_info

    # 5. 파일 저장
    # 'w' 모드는 파일이 없으면 자동으로 생성하고, 있으면 덮어씁니다.
    with open(dhcp_setting_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    return dhcp_info


# 프로젝트 DHCP setting json 파일에서 특정 라우터/스위치의 DHCP 세팅을 복원하는 함수
def restore_dhcp_setting(con: BaseConnection, 
                         dhcp_setting_json_path: str) -> list:
    with open(dhcp_setting_json_path, 'r', encoding="utf-8") as f:
        data = json.load(f)

    device_name = get_host_name(con)

    role = get_device_role(con)
    dhcp_pools = data[device_name]
    cmds = []

    # 1. 스위치일 경우 DHCP 서비스 활성화 명령 추가
    if role == "SWITCH":
        cmds.append("service dhcp")

    # 2. 제외 주소(Excluded Addresses) 복구
    # 여러 풀에 중복된 제외 주소가 있을 수 있으므로 set으로 중복 제거 후 추가
    processed_excl = set()
    for pool in dhcp_pools:
        for excl in pool.get("Excluded_Addresses", []):
            ex_key = (excl['start'], excl['end'])
            if ex_key not in processed_excl:
                cmds.append(f"ip dhcp excluded-address {excl['start']} {excl['end']}")
                processed_excl.add(ex_key)

    # 3. 각 Pool 상세 정보 복구
    for pool in dhcp_pools:
        cmds.append(f"ip dhcp pool {pool['Poolname']}")
        if pool['NID'] and pool['Netmask']:
            cmds.append(f" network {pool['NID']} {pool['Netmask']}")
        if pool['Gateway']:
            cmds.append(f" default-router {pool['Gateway']}")
        if pool['DNS_Server_IP']:
            cmds.append(f" dns-server {pool['DNS_Server_IP']}")

    # 4. 명령어 실행
    if len(cmds) > 0:
        run_conf(con, cmds)
    
    return dhcp_pools

# 라우터 NAT(PAT)의 설정 정보를 가져오는 함수
def get_nat_pat_config(con: BaseConnection) -> dict:
    # 1. 장비에 명령어 전송 
    nat_stats = run_cmd(con, "sh ip nat statistics")
    acl_output = run_cmd(con, "sh access-list")

    in_interface = ""
    out_interface = ""
    
    # 2. NAT 인터페이스 추출
    nat_lines = nat_stats.splitlines()
    for i in range(len(nat_lines)):
        line = nat_lines[i].strip()
        # 'Inside' 단어가 포함된 줄을 찾고 그 다음 줄 데이터 확보
        if "Inside interfaces:" in line and i + 1 < len(nat_lines):
            in_interface = nat_lines[i+1].strip()
        # 'Outside' 단어가 포함된 줄을 찾고 그 다음 줄 데이터 확보
        elif "Outside interfaces:" in line and i + 1 < len(nat_lines):
            out_interface = nat_lines[i+1].strip()

    # 3. ACL 및 Prefix 계산
    ip_and_prefix = []
    acl_lines = acl_output.splitlines()
    
    for line in acl_lines:
        if "permit" in line and "wildcard bits" in line:
            # 'permit' 문자열 이후부터 ',' 전까지가 IP 주소
            parts = line.split("permit")
            remaining = parts[1].split(",")
            ip = remaining[0].strip()
            
            # 'wildcard bits' 뒤에 오는 것이 마스크 값
            wildcard = line.split("wildcard bits")[1].strip()

            
            # 각 옥텟을 이진수로 바꿔서 '1'의 개수를 더하기
            total_host_bits = 0
            for octet in wildcard.split('.'):
                # bin()은 '0b1111' 형태를 반환하므로 .count('1')로 비트 수 측정
                total_host_bits += bin(int(octet)).count('1')
            
            # 전체 32비트 - 호스트 비트 수 (Prefix)
            prefix = 32 - total_host_bits
            ip_and_prefix.append(f"{ip}/{prefix}")

    # 4. 최종 결과 반환
    return {
        "IN_Interface": in_interface,
        "OUT_Interface": out_interface,
        "IP_Netmask_Prefix": ip_and_prefix[0]
    }

# NAT(PAT) 세팅 json에 저장하는 함수
def store_nat_pat_setting(con: BaseConnection, 
                      nat_setting_json_path: str) -> dict:
    """
    현재 라우터의 NAT 설정을 읽어 JSON 파일에 저장
    """
    # 1. 현재 라우터의 NAT 설정 정보 가져오기
    nat_config = get_nat_pat_config(con)
    
    # 2. 데이터 구조 초기화 및 기존 파일 읽기
    data = {}
    if os.path.exists(nat_setting_json_path) and os.path.getsize(nat_setting_json_path) > 0:
        with open(nat_setting_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)

    # 3. 데이터 업데이트 (장비 호스트명을 키로 저장)
    data[get_host_name(con)] = nat_config

    # 4. 파일 저장
    with open(nat_setting_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    return nat_config

# NAT(PAT) 세팅 json에서 복구하는 함수
def restore_nat_pat_setting(con: BaseConnection, 
                        nat_setting_json_path: str) -> dict:
    if not os.path.exists(nat_setting_json_path):
        return {}
    if device_name not in data or not data[device_name]:
        return
    data = None
    with open(nat_setting_json_path, 'r', encoding="utf-8") as f:
        data = json.load(f)

    device_name = get_host_name(con)

    config = data[device_name]
    
    return router_pat_nat_config(con, in_interface=config["IN_Interface"], out_interface=config["OUT_Interface"], ip_and_netmask_prefix=config["IP_Netmask_Prefix"])

In [17]:
"""
1. STP
    1-1. STP 설정 O  : switch_stp_config  -----------test
    1-2. 정보 확인 O : get_stp_info  -----------test
    1-3. 백업 - 어떤 형태로 저장할지... ???
    1-4. rollback ??? 
2. VLAN
    2-0. run_vlan -> vlan 설정 모드에서 cmds실행 ---------------- test
    2-1. VLAN 설정   :   switch_vlan_range_config, switch_trunk_range_config
    2-2. 정보 확인   :   get_vlan_port_dict
    2-3. 백업 - 어떤 형태로 저장할지... :   store_vlan_port_data
    2-4. rollback  :  rstore_vlan_port_data
3. VTP
   
    3-1. vtp 모드 + domain + password 설정 O : switch_vtp_config -----------test
    3-2. vtp 정보 확인 O : get_stp_info
    3-3. vtp 백업 O - store_vtp_config -------------------test
    3-4. rollback O : restore_vtp_config
    
4. Inter Vlan
    4-1. 설정 O : inter_vlan_config -----------------------test
    4-2. 백업 O : store_vlan_interfaces ---------------- test 
    4-3. rollback O : restore_vlan_interfaces
    
    
5. NAT (Static, dynamic) - 설정
    5-1. static O : router_static_nat_config
    5-2. dynamic O : router_dynamic_nat_config

SyntaxError: EOF while scanning triple-quoted string literal (888325690.py, line 27)

In [9]:
#stp

#STP 설정 
def switch_stp_config(con: BaseConnection,
                       vlan : str,
                       priority : str) -> str:
    
    if (priority % 4096 != 0):
        return "STP 설정 실패 : 잘못된 priority 값:"
        
    cmds=[
        f"spanning-tree vlan {vlan} priority {priority}"
    ]
    return run_conf(con, cmds)


#STP 정보 확인 
#스위치 정보 
def get_stp_info(con: BaseConnection) -> str:
    cmd= "sh spanning-tree bri" 
    return run_cmd(con, cmd)




SyntaxError: EOL while scanning string literal (2956104153.py, line 9)

In [10]:
#vlan

In [11]:
#VTP

In [12]:
#VTP mode 설정 (mode + domain + password)
def switch_vtp_config(con: BaseConnection,
                      domain_name: str,
                      password: str,
                      vtp_mode: str = "server"
                     ) -> str:
    cmds = [
        f"vtp {vtp_mode}",
        f"vtp domain {domain_name}",
        f"vtp password {password}"
    ]
    return run_vlan(con, cmds)

#vtp 정보 확인
def get_stp_info(con: BaseConnection) -> str:
    cmd= "sh vtp status" 
    return run_cmd(con, cmds)

#vtp 백업
#vtp 백업
import os
import json
from netmiko import BaseConnection

def get_vtp_config(con: BaseConnection) -> dict:

    vtp_status = con.send_command("show vtp status")
    
    vtp_domain = "Unknown"
    vtp_mode = "Server"
    
    # 줄바꿈 단위로 쪼개서 필요한 정보 매칭
    for line in vtp_status.splitlines():
        line = line.strip()
        
        # 1. VTP Domain Name 추출 (예: VTP Domain Name : soongsil)
        if "VTP Domain Name" in line:
            parts = line.split(":")
            if len(parts) > 1:
                vtp_domain = parts[1].strip()
                
        # 2. VTP Operating Mode 추출 (예: VTP Operating Mode : Server)
        if "VTP Operating Mode" in line:
            parts = line.split(":")
            if len(parts) > 1:
                vtp_mode = parts[1].strip().lower() # 복구 편의를 위해 소문자(server/client)로 통일
                
    # 장비에서 비번은 못 읽으므로 일단 가이드 텍스트를 기본값으로 둡니다.
    return {
        "domain": vtp_domain,
        "mode": vtp_mode,
        "password": "null"
    }


def store_vtp_config(con: BaseConnection, vtp_json_path: str, custom_password: str = None) -> dict:
  
    # 스위치 이름(hostname) 알아내기
    raw_config = con.send_command("show running-config")
    switch_name = "Unknown_Switch"
    for line in raw_config.splitlines():
        line = line.strip()
        if line.startswith("hostname "):
            switch_name = line.split()[1] # "hostname ESW1" -> "ESW1" 가져옴
            break
            
    # 장비의 VTP 도메인, 모드 파싱 정보 가져오기
    vtp_info = get_vtp_config(con)
    
    # 함수 부를 때 실제 비번 넣어주면 그걸로 덮어쓰기 합니다.
    if custom_password:
        vtp_info["password"] = custom_password
        
    # 데이터 구조 초기화 및 기존 파일 읽기
    data = {}
    if os.path.exists(vtp_json_path) and os.path.getsize(vtp_json_path) > 0:
        with open(vtp_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)
            
    # 장비 이름을 Key로 잡고 파싱한 데이터 집어넣기
    data[switch_name] = vtp_info
    
    # 5. 파일 정렬 후 저장 (indent=4로 예쁘게 들여쓰기!)
    with open(vtp_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    
    return data

In [13]:
#Router - inter vlan 설정
#설정
#inter vlan 설정
def inter_vlan_config(con: BaseConnection,
                      interface: str,
                      vlan: str,
                      ip: str,
                      netmask: str) -> str:
    cmds = [
        f"int {interface}.{vlan}",
        f"enc dot1Q {vlan}",
        f"ip add {ip} {netmask}",
        "exit",
        f"int {interface}",
        "no sh"
    ]
    return run_conf(con, cmds)

#백업

#inter vlan 
def get_vlan_interfaces(con: BaseConnection) -> dict:

    # 라우터에서 sh run 명령어 실행
    raw_config = con.send_command("show running-config")
    
    router_name = "Unknown_Router" #default name
    router_backup = {}
    
    # \n 를 기준으로 전체 텍스트 쪼개기
    lines = raw_config.splitlines()
    
    # 먼저 첫 번째 패스로 라우터 이름(hostname)부터 찾기
    for line in lines:
        line = line.strip()
        if line.startswith("hostname "):
            router_name = line.split()[1] # "hostname R1" 에서 "R1"만 쏙 가져옴
            break
            
    # 라우터 이름을 키로 가진 딕셔너리 리스트 생성
    router_backup[router_name] = []
    
    # 3. 가상 인터페이스 및 IP 정보 추적 시작
    current_iface = None
    vlan_id = None
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # 가상 인터페이스 구역 진입 확인 (예: interface FastEthernet0/0.10)
        if line.startswith("interface ") and "." in line:
            current_iface = line.split()[1]           # 두 번째 단어인 인터페이스명 가져오기
            vlan_id = current_iface.split(".")[-1]      # 소수점 뒷자리를 VLAN ID로 지정
            
            # 딕셔너리 뼈대 생성해서 넣어두기
            iface_info = {
                "interface": current_iface,
                "vlan": vlan_id,
                "ip": "",
                "netmask": ""
            }
            router_backup[router_name].append(iface_info)
            continue
            
        # 일반 물리 포트 인터페이스를 만나면 반복문으로 돌아가기
        elif line.startswith("interface ") and "." not in line:
            current_iface = None
            vlan_id = None
            continue
            
        # 가상 포트 구역 내부  +  ip address 라인을 만났을 때
        if current_iface and line.startswith("ip address "):
            parts = line.split()
            # parts 예시: ['ip', 'address', '10.0.10.1', '255.255.255.0']
            if len(parts) >= 4:
                ip_addr = parts[2]
                netmask_addr = parts[3]
                
                # 가장 마지막에 리스트에 추가된 가상 포트 딕셔너리에 데이터 채우기
                router_backup[router_name][-1]["ip"] = ip_addr
                router_backup[router_name][-1]["netmask"] = netmask_addr
                
    return router_backup

def store_vlan_interfaces(con: BaseConnection, 
                      vlan_int_setting_json_path: str) -> dict:
    """
    현재 라우터의 NAT 설정을 읽어 JSON 파일에 저장
    """
    # 1. 현재 라우터의 NAT 설정 정보 가져오기
    vlan_int_config = get_vlan_interfaces(con)
    
    # 2. 데이터 구조 초기화 및 기존 파일 읽기
    data = {}
    if os.path.exists(vlan_int_setting_json_path) and os.path.getsize(vlan_int_setting_json_path) > 0:
        with open(vlan_int_setting_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)

    # 3. 데이터 업데이트 (장비 호스트명을 키로 저장)
    data.update(vlan_int_config)

    # 4. 파일 저장
    with open(vlan_int_setting_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    return vlan_int_config

In [14]:
#NAT 설정

#static
def router_static_nat_config(con: BaseConnection,
                      in_interface: str,
                      out_interface: str,
                      in_addr: str,
                      out_addr: str) -> str:
    cmds = [
        f"interface {in_interface}",
        "ip nat inside",
        f"interface {out_interface}",
        "ip nat outside",
        f"ip nat inside source static {in_addr} {out_addr}"
    ]
    return run_conf(con, cmds)

#dynamic
def router_dynamic_nat_config(con: BaseConnection,
                      in_interface: str,
                      out_interface: str,
                      ip_and_netmask_prefix: str,
                      nat_ip_pool_name: str,
                      nat_ip_pool_start_ip: str,
                      nat_ip_pool_end_ip: str,
                      nat_ip_pool_netmask: str) -> str:
    cmds = [
        f"interface {in_interface}",
        "ip nat inside",
        f"interface {out_interface}",
        "ip nat outside",
        f"ip nat pool {nat_ip_pool_name} {nat_ip_pool_start_ip} {nat_ip_pool_end_ip} netmask {nat_ip_pool_netmask}",
        f"access-list 1 permit {_wildcard_(ip_and_netmask_prefix)}",
        f"ip nat inside source list 1 pool {nat_ip_pool_name}"
    ]
    return run_conf(con, cmds)

In [5]:
import paramiko
import time

def init_ubuntu_package(ip, username="root", password="rifkehtm!@", port=22):
    # 접속할 우분투 서버 정보 입력
    SERVER_IP = ip#"172.16.124.3"
    USERNAME = username#"root"
    PASSWORD = password#"rifkehtm123!@"
    PORT = port
    #로컬 만든 후 루트 아이디까지 만들고 나서 실행하기
    #우분투는 root로 로그인한다고 가정한다.
    
    # 우분투 서버에서 한 번에 실행될 쉘 스크립트 (무인 모드 적용)
    setup_commands = """
    
    add-apt-repository universe -y  
    apt update -y && apt upgrade -y
    apt install -y openssh-server wget vim libstdc++6 tar gzip build-essential iputils-ping ufw
    
    
    systemctl enable --now ssh
    ufw allow ssh
    ufw allow 8080
    ufw disable
    
    wget https://www.ubiedu.co.kr/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    chmod u+x /tmp/miniconda.sh && /tmp/miniconda.sh -b -u -p /root/miniconda3
    /root/miniconda3/bin/conda init bash
    sed -i '$ a export PATH=$PATH:/root/miniconda3/bin' /etc/bashrc
    /root/miniconda3/bin/conda config --set auto_activate false
    /root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    /root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    /root/miniconda3/bin/conda install jupyter notebook -y
    /root/miniconda3/bin/pip install notebook
    expect -c "spawn /root/miniconda3/bin/jupyter notebook password; expect \\"Enter password:\\"; send \\"asd123!@\\r\\"; expect \\"Verify password:\\"; send \\"asd123!@\\r\\"; expect eof"
    sudo tee /etc/systemd/system/jupyter.service > /dev/null <<EOF
    [Unit]
    Description=Jupyter Notebook Service
    After=network.target
    
    [Service]
    Type=simple
    User=root
    Group=root
    WorkingDirectory=/root
    ExecStart=/root/miniconda3/bin/jupyter notebook --allow-root --ip=0.0.0.0 --port=80 --no-browser
    Restart=always
    RestartSec=10
    
    [Install]
    WantedBy=multi-user.target
    EOF
    systemctl daemon-reload && systemctl enable --now jupyter
    /root/miniconda3/bin/conda install pip -y && /root/miniconda3/bin/pip install --upgrade pip && /root/miniconda3/bin/pip install paramiko
    /root/miniconda3/bin/python3 -m pip install ipykernel && /root/miniconda3/bin/python3 -m ipykernel install --user --name system-python --display-name 'Python 3 (System)'
    rm -f /tmp/miniconda.sh
    
    """
    
    # 이게 실행되어야함
    setup_commands = """
    add-apt-repository universe -y  
    apt update -y && apt upgrade -y
    apt install -y openssh-server wget vim libstdc++6 tar gzip build-essential iputils-ping ufw expect
    
    systemctl enable --now ssh
    ufw allow ssh
    ufw allow 80
    ufw allow 8888
    ufw disable
    
    wget https://www.ubiedu.co.kr/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    chmod u+x /tmp/miniconda.sh
    /tmp/miniconda.sh -b -u -p /root/miniconda3
    rm -f /tmp/miniconda.sh
    
    /root/miniconda3/bin/conda init bash
    sed -i '$ a export PATH=$PATH:/root/miniconda3/bin' ~/.bashrc
    /root/miniconda3/bin/conda config --set auto_activate false
    /root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    /root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    
    /root/miniconda3/bin/conda install jupyter notebook -y
    /root/miniconda3/bin/pip install notebook paramiko ipykernel
    
    expect -c "
    spawn /root/miniconda3/bin/jupyter notebook password
    expect \\"Enter password:\\"
    send \\"1234\\r\\"
    expect \\"Verify password:\\"
    send \\"1234\\r\\"
    expect eof
    "
    
    cat <<EOF > /etc/systemd/system/jupyter.service
    [Unit]
    Description=Jupyter Notebook Service
    After=network.target
    
    [Service]
    Type=simple
    User=root
    Group=root
    WorkingDirectory=/root
    ExecStart=/root/miniconda3/bin/jupyter notebook --allow-root --ip=0.0.0.0 --port=8888 --no-browser
    Restart=always
    RestartSec=10
    
    [Install]
    WantedBy=multi-user.target
    EOF
    
    systemctl daemon-reload
    systemctl enable --now jupyter
    /root/miniconda3/bin/python3 -m ipykernel install --user --name system-python --display-name 'Python 3 (System)'
    """
    
    print(f"[{SERVER_IP}] 서버에 SSH 연결을 시도합니다...")
    
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    client.connect(SERVER_IP, port=PORT, username=USERNAME, password=PASSWORD, timeout=10)
    print("연결 성공! 자동 설정 스크립트를 원격으로 실행합니다. (몇 분 정도 소요됩니다)\n")
    
    # 명령어 실행 및 실시간 출력
    stdin, stdout, stderr = client.exec_command(setup_commands)
    
    for line in iter(stdout.readline, ""):
        print(line, end="")
    
    # 에러 메시지가 있다면 별도 출력
    error_msg = stderr.read().decode()
    if error_msg:
        print("\n[오류 또는 경고 메시지]:")
        print(error_msg)
    
    client.close()
    print("\nSSH 연결이 안전하게 종료되었습니다.")

In [6]:
import paramiko
import argparse
import sys
import time

COMMANDS = [
    ("필수 패키지 설치", "dnf -y install epel-release wget vim libstdc++ tar gzip"),
    ("expect 설치", "dnf makecache && dnf install -y expect"),
    ("CRB 레포지토리 활성화", "dnf config-manager --set-enabled crb"),
    ("시스템 업데이트 및 업그레이드", "dnf -y update && dnf -y upgrade"),
    ("SELinux 비활성화", "sed -i 's/^SELINUX=enforcing/SELINUX=disabled/' /etc/selinux/config && setenforce 0"),
    ("방화벽 중지", "systemctl disable --now firewalld"),
    ("Miniconda 다운로드", "wget https://www.ubiedu.co.kr/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh"),
    ("Miniconda 무인 설치", "chmod u+x /tmp/miniconda.sh && /tmp/miniconda.sh -b -u -p /root/miniconda3"),
    ("Conda 초기화", "/root/miniconda3/bin/conda init bash"),
    ("환경변수 등록", "sed -i '$ a export PATH=$PATH:/root/miniconda3/bin' /etc/bashrc"),
    ("Conda 자동활성화 방지", "/root/miniconda3/bin/conda config --set auto_activate false"),
    
    # ToS 에러 방지: main 및 r 채널 약관 동의 추가
    ("Conda 약관 동의 (Main)", "/root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main"),
    ("Conda 약관 동의 (R)", "/root/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r"),
    
    # 주피터와 노트북 엔진을 명시적으로 함께 설치
    ("주피터 패키지 설치", "/root/miniconda3/bin/conda install jupyter notebook -y"),
    
    # Conda 설치 누락을 대비한 Pip 강제 설치 (가장 확실한 방법)
    ("주피터 엔진 최종 확인", "/root/miniconda3/bin/pip install notebook"),
    
    ("주피터 비밀번호 설정 (expect)", 'expect -c "spawn /root/miniconda3/bin/jupyter notebook password; expect \\"Enter password:\\"; send \\"asd123!@\\r\\"; expect \\"Verify password:\\"; send \\"asd123!@\\r\\"; expect eof"'),
    
    ("주피터 systemd 서비스 등록", '''
sudo tee /etc/systemd/system/jupyter.service > /dev/null <<EOF
[Unit]
Description=Jupyter Notebook Service
After=network.target

[Service]
Type=simple
User=root
Group=root
WorkingDirectory=/root
ExecStart=/root/miniconda3/bin/jupyter notebook --allow-root --ip=0.0.0.0 --port=80 --no-browser
Restart=always
RestartSec=10

[Install]
WantedBy=multi-user.target
EOF
'''),
    ("주피터 서비스 시작", "systemctl daemon-reload && systemctl enable --now jupyter"),
    ("Pip 및 라이브러리 설치", "/root/miniconda3/bin/conda install pip -y && /root/miniconda3/bin/pip install --upgrade pip && /root/miniconda3/bin/pip install paramiko"),
    ("커널 등록", "/root/miniconda3/bin/python3 -m pip install ipykernel && /root/miniconda3/bin/python3 -m ipykernel install --user --name system-python --display-name 'Python 3 (System)'"),
    ("임시 파일 삭제", "rm -f /tmp/miniconda.sh")
]

def get_ssh_client(ip: str, password: str, username: str, port: int) -> paramiko.SSHClient:
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    print(f"\n{username}@{ip}:{port}에 SSH 접속 시도 중..")
    client.connect(hostname=ip, port=port, username=username, password=password)
    return client

def setup(client: paramiko.SSHClient):
    global COMMANDS
    for desc, com in COMMANDS:
        print(f"\n[{desc}] 시작...")
        stdin, stdout, stderr = client.exec_command(com)
        
        for line in iter(stdout.readline, ""):
            print(f"  {line.strip()}")
            
        # 에러 및 로그 확인
        error = stderr.read().decode("utf-8")
        if error:
            # 경고 메시지도 stderr에 찍히므로 에러라고 단정짓기보다 로그로 표시
            print(f"\n[시스템 메시지]:\n{error}")

def init_rocky_package(ip, user="root", pw="asd123!@", port=22):
    client = get_ssh_client(ip, pw, user, port)
    setup(client)
    client.close()

# 260601 Rocky DHCP 서버 셋업
def rocky_dhcp_server_setup(
    ip: str, root_password: str,
    domain_name: str, dns1: str, dns2: str,
    default_lease: int, max_lease: int,
    subnet_id: str, netmask: str,
    range_start: str, range_end: str,
    gateway: str, broadcast: str
):
    # > 리다이렉션을 사용하여 기존 파일 내용을 완전히 지우고 새로 씁니다.
    dhcp_conf_script = f"""cat << 'EOF' > /etc/dhcp/dhcpd.conf
option domain-name "{domain_name}";
option domain-name-servers {dns1}, {dns2};
default-lease-time {default_lease};
max-lease-time {max_lease};

authoritative;

subnet {subnet_id} netmask {netmask} {{
  range {range_start} {range_end};
  option routers {gateway};
  option broadcast-address {broadcast};
}}
EOF"""

    CMD = [
        ("DHCP 패키지 설치", "dnf install dhcp-server -y"),
        ("기존 설정파일 초기화 및 신규 생성", dhcp_conf_script), # 백업 라인 제거됨
        ("DHCP 서비스 시작 및 활성화", "systemctl enable --now dhcpd")
    ]

    client = get_ssh_client(ip, root_password, "root", 22)
    run_cmd(client, CMD)

In [26]:
import paramiko
import argparse
import sys
import time

def get_ssh_client(ip: str, password: str, username: str, port: int) -> paramiko.SSHClient:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        print(f"\n{username}@{ip}:{port}에 SSH 접속 시도 중..")
        client.connect(hostname=ip, port=port, username=username, password=password)
        return client

def run_cmd(client: paramiko.SSHClient, cmd):
        for desc, com in cmd:
            print(f"\n[{desc}] 시작...")
            stdin, stdout, stderr = client.exec_command(com)
            
            for line in iter(stdout.readline, ""):
                print(f"  {line.strip()}")
                
            # 에러 및 로그 확인
            error = stderr.read().decode("utf-8")
            if error:
                # 경고 메시지도 stderr에 찍히므로 에러라고 단정짓기보다 로그로 표시
                print(f"\n[시스템 메시지]:\n{error}")

def adduser_to_host(ip: str, root_password: str, new_user_name: str, new_user_passwd: str):
    CMD = [
        ("유저 만들기", f"useradd -m {new_user_name}"),
        ("유저 비밀번호 설정", f"echo '{new_user_name}:{new_user_passwd}' | chpasswd")
    ]

    run_cmd(get_ssh_client(ip, root_password, "root", 22), CMD)


def install_rocky_httpd(ip: str, root_password: str, user_name: str, new_user_passwd: str, is_user: bool, index_html_path: str=None):
    NEW_USER = [
        ("유저 만들기", f"useradd -m {user_name}"),
        ("유저 비밀번호 설정", f"echo '{user_name}:{new_user_passwd}' | chpasswd")
    ]
    
    indexhtml = None
    if index_html_path != None:
        # TODO: html 읽어다 문자열에 저장
        with open(index_html_path, "r", encoding="utf-8") as f:
            indexhtml = f.read()

    COMMANDS = [
        ("유저 홈 디렉터리 권한 변경", f"chmod -R 777 /home/{user_name}"),
        ("httpd 설치", "dnf install -y httpd"),
        ("httpd 실행", "systemctl enable --now httpd"),
        ("기존 DocumentRoot 주석 처리", f"sed -i 's|^DocumentRoot \"/var/www/html\"|#DocumentRoot \"/var/www/html\"|' /etc/httpd/conf/httpd.conf"),
        ("아랫줄에 새 DocumentRoot 추가", f"sed -i '/#DocumentRoot \"\/var\/www\/html\"/a DocumentRoot \"/home/{user_name}\"' /etc/httpd/conf/httpd.conf"),
        ("파일 맨 끝에 새로운 Directory 설정 블록 추가", f"""echo -e '\n<Directory "/home/{user_name}">\n    Options Indexes FollowSymLinks\n    AllowOverride None\n    Require all granted\n</Directory>' >> /etc/httpd/conf/httpd.conf"""),
        ("httpd 설정 재적용", "systemctl restart httpd")
    ]

    if is_user == False:
        COMMANDS = NEW_USER + COMMANDS

    if index_html_path != None:
        # HTML 내용에 공백이나 줄바꿈이 있어도 깨지지 않도록 리눅스 single quote(')로 감싸서 echo 하도록 수정했습니다.
        COMMANDS = COMMANDS + [("index.html 파일 작성해주기", f"echo '{indexhtml}' > /home/{user_name}/index.html")]
        
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)

def add_virtual_host_to_httpd(ip: str, root_password: str, vhostip: str, vhostport: str, vhostfolderpath: str):
    # 멀티라인 개행과 기호들이 파라미코 환경에서 깨지지 않도록 한 줄짜리 echo -e 명령어로 변환
    vhost_config = (
        fr"\n<VirtualHost {vhostip}:{vhostport}>\n"
        fr'    DocumentRoot "{vhostfolderpath}"\n'
        fr'    <Directory "{vhostfolderpath}">\n'
        fr"        Require all granted\n"
        fr"    </Directory>\n"
        fr"</VirtualHost>"
    )

    run_cmd(get_ssh_client(ip, root_password, "root", 22), [
         # 이미 폴더가 있으면 [시스템 메시지]에 File exists가 뜨지만 아래 세팅으로 정상 진행됩니다.
         ("디렉터리 만들기(없다면)", f"mkdir {vhostfolderpath}"),
         
         # 이 한 줄 명령어가 vhost.conf 맨 뒤에 이쁘게 내용을 추가해 줍니다.
         ("vhost.conf 세팅", f'echo -e "{vhost_config}" >> /etc/httpd/conf.d/vhost.conf')
    ])

# MariaDB 셋업
def rocky_setup_mariadb(ip: str, 
                        root_password: str, 
                        maria_db_pasword: str):
    CMD = [
        # 1. OS 및 저장소 세팅
        ("dnf 업데이트", "dnf update -y"),
        ("dnf 업그레이드", "dnf upgrade -y"),
        ("epel 설치", "dnf install epel-release -y"),
        ("remi 설치", "dnf install https://rpms.remirepo.net/enterprise/remi-release-9.rpm -y"),
        ("remi 활성화", "dnf config-manager --set-enabled remi"),
        
        # 2. PHP 세팅
        ("php 모듈 설정 초기화", "dnf module reset php -y"),
        ("php 8.4 기본 설정 스트림으로 예약", "dnf module enable php:remi-8.4 -y"),
        ("php 연동 모듈 통합 설치", "dnf install -y php php-cli php-fpm php-common php-devel php-mbstring php-odbc php-process php-gd php-mysqlnd php-pear php-xml php-xmlrpc php-zip php-opcache php-intl php-pdo php-bcmath gcc gcc-c++ gdbm-devel zlib-devel libzip-devel libjpeg-devel libpng-devel freetype-devel gd-devel giflib-devel"),
        
        # 3. 테스트 파일 생성 (vim 대체)
        ("phpinfo 페이지 생성", "echo '<?php phpinfo(); ?>' > /var/www/html/phpinfo.php"),
        
        # 4. MariaDB 세팅
        ("MariaDB 설치", "dnf -y install mariadb-server"),
        ("MariaDB 기본 문자셋(utf8mb4) 설정", "echo -e '[mysqld]\ncharacter-set-server=utf8mb4\ncollation-server=utf8mb4_unicode_ci\n[client]\ndefault-character-set=utf8mb4' > /etc/my.cnf.d/mariadb-server.cnf"),
        ("MariaDB 자동 실행 및 시작", "systemctl enable --now mariadb"),
        
        # 5. MariaDB 초기 보안 설정 (mysql_secure_installation 대체)
        # 쉘에서 SQL 명령어를 직접 주입하여 root 비밀번호 변경 및 익명 계정 삭제 처리
        ("MariaDB 초기 보안 설정 적용", f"mysql -e \"ALTER USER 'root'@'localhost' IDENTIFIED BY '{maria_db_pasword}'; DELETE FROM mysql.user WHERE User=''; DROP DATABASE IF EXISTS test; FLUSH PRIVILEGES;\""),
        
        # 6. phpMyAdmin 세팅
        ("phpMyAdmin 설치", "dnf -y install phpmyadmin"),
        ("phpMyAdmin 소유권 변경", "chown -R apache:apache /usr/share/phpMyAdmin/"),
        # 기본적으로 로컬(localhost) 접속만 허용되어 있으므로 외부 IP에서도 접속 가능하게 변경 (vim 대체)
        ("phpMyAdmin 외부 접속 허용 설정", "sed -i 's/Require local/Require all granted/g' /etc/httpd/conf.d/phpMyAdmin.conf"),
        
        # 7. 서비스 최종 구동
        ("서버 자동 실행 등록", "systemctl enable --now httpd php-fpm"),
        ("웹 서버 및 PHP-FPM 재시작", "systemctl restart httpd php-fpm mariadb")
    ]

    run_cmd(get_ssh_client(ip, root_password, "root", 22), CMD)



# 260601 추가
# DNS(named) 설치 후 로그 남기기 설정
def rocky_install_named_and_logging(ip: str, 
                                    root_password: str):
    named_conf_script = """
    if ! grep -q 'channel query_log' /etc/named.conf; then
        # 1. 기존의 기본 logging 블록 삭제 (logging { 부터 }; 까지)
        sed -i '/^logging[[:space:]]*{/,/^};/d' /etc/named.conf
        
        # 2. 새로운 커스텀 로깅 블록 추가 (EOF 활용)
        cat >> /etc/named.conf << 'EOF'

logging {
    channel default_debug {
        file "data/named.run";
        severity dynamic;
    };
    channel query_log {
        file "/var/log/named/query.log" versions 5 size 50M;
        severity info;
        print-time yes;
        print-category yes;
        print-severity yes;
    };
    channel security_log {
        file "/var/log/named/security.log" versions 3 size 50M;
        severity warning;
        print-time yes;
        print-category yes;
        print-severity yes;
    };
    category queries { query_log; };
    category security { security_log; };
    category default { default_debug; };
};
EOF
        echo "✅ named.conf에 커스텀 로깅 설정이 성공적으로 추가되었습니다."
    else
        echo "✅ 이미 로깅 설정이 적용되어 있습니다. (수정 건너뜀)"
    fi
    """

    CMD = [
        ("OS 패키지 업데이트", "dnf update -y"),
        # Rocky Linux의 DNS 패키지 공식 명칭은 'bind'와 'bind-utils' 입니다.
        ("BIND(named) 설치", "dnf install bind bind-utils -y"),
        ("로그 폴더 생성", "mkdir -p /var/log/named"),
        ("로그 폴더 소유권 변경", "chown named:named /var/log/named"),
        ("로그 폴더 권한 통제", "chmod 750 /var/log/named"),
        # 위에서 작성한 스크립트를 서버로 전송하여 실행
        ("named.conf 로그 설정 병합", named_conf_script),
        # 설정 파일 오타/문법 검사 (통과하면 아무 메시지도 안 뜸)
        ("설정 파일 문법 검증", "named-checkconf"),
        # 서비스 즉시 실행 및 재부팅 시 자동 실행 등록
        ("DNS 서비스 시작 및 활성화", "systemctl enable --now named")
    ]
    run_cmd(get_ssh_client(ip, root_password, "root", 22), CMD)

In [8]:
import paramiko
import argparse
import sys
import time


def install_ubuntu_nginx(ip: str, root_password: str, user_name: str, new_user_passwd: str, is_user: bool, index_html_path: str=None):
    # [우분투] 유저 생성 및 디렉터리 세팅
    NEW_USER = [
        ("유저 만들기", f"useradd -m {user_name}"),
        ("유저 비밀번호 설정", f"echo '{user_name}:{new_user_passwd}' | chpasswd"),
        ("유저 홈 디렉터리 강제 생성", f"mkdir -p /home/{user_name}")
    ]
    
    indexhtml = "test"  # 기본값 "test"
    if index_html_path != None:
        with open(index_html_path, "r", encoding="utf-8") as f:
            indexhtml = f.read()

    # [우분투 + Nginx] 메인 실행 명령어 리스트
    COMMANDS = [
        ("패키지 업데이트", "apt update"),
        # 💡 [복구 단계 추가] 기존에 깨진 패키지 잠금을 해제하고 자동 복구합니다.
        ("dpkg 잠금 및 오류 강제 복구", "dpkg --configure -a"),
        # 💡 [설치 안정화] 자동화 스크립트 튕김 방지 옵션을 추가하여 nginx를 설치합니다.
        ("nginx 설치", "DEBIAN_FRONTEND=noninteractive apt -y install nginx"),
        ("nginx 실행", "systemctl enable --now nginx"),
        ("방화벽 비활성화", "ufw disable"),
        ("Nginx DocumentRoot 경로 수정", f"sed -i 's|root /var/www/html;|root /home/{user_name};|' /etc/nginx/sites-available/default"),
    ]

    if is_user == False:
        COMMANDS = NEW_USER + COMMANDS

    # index.html 파일 작성 및 권한 설정을 리스트에 결합
    COMMANDS = COMMANDS + [
        ("index.html 파일 작성해주기", f"echo '{indexhtml}' > /home/{user_name}/index.html"),
        ("모든 권한 설정 (777)", f"chmod -R 777 /home/{user_name}"),
        ("소유권 이동", f"chown -R {user_name}:{user_name} /home/{user_name}"),
        ("nginx 설정 검사", "nginx -t"),
        ("nginx 설정 재적용", "systemctl restart nginx")
    ]
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)

def add_virtual_host_to_nginx(ip: str, root_password: str, vhostip: str, vhostport: str, vhostfolderpath: str, domain_name: str = "_"):
    """
    Nginx 가상 호스트(Server Block) 설정 함수
    domain_name: 특정 도메인이 없다면 기본값 "_" (모든 호스트 수신)로 설정됩니다.
    """
    # 1. Nginx 문법에 맞는 Server 블록 구성
    # Nginx는 아파치와 달리 listen에 IP:포트를 적거나 포트만 적고 server_name으로 도메인을 구별합니다.
    nginx_config = (
        fr"server {{\n"
        fr"    listen {vhostport};\n"
        fr"    server_name {domain_name};\n\n"
        fr"    root {vhostfolderpath};\n"
        fr"    index index.html index.htm;\n\n"
        fr"    location / {{\n"
        fr"        try_files \$uri \$uri/ =404;\n"
        fr"    }}\n"
        fr"}}"
    )

    # Nginx는 도메인명이나 포트명을 따서 개별 파일로 저장하는 것이 나중에 관리하기 좋습니다.
    config_filename = f"vhost_{vhostport}.conf"

    run_cmd(get_ssh_client(ip, root_password, "root", 22), [
         # [작업 1] 웹 서비스 디렉터리 생성 (이미 있으면 에러가 뜨지만 다음으로 넘어감)
         ("디렉터리 만들기(없다면)", f"mkdir {vhostfolderpath}"),
         ("디렉터리 권한 777", f"chmod 777 -R {vhostfolderpath}"),
            ("index.html 생성", f"touch {vhostfolderpath}/index.html"),
         
         # [작업 2] Nginx는 이어붙이기(>>)가 아니라 개별 파일 새로 쓰기(>)를 씁니다.
         # 기존 설정을 오염시키지 않고 깔끔하게 독립된 파일이 생성됩니다.
         ("Nginx 블록 세팅", f'echo -e "{nginx_config}" > /etc/nginx/conf.d/{config_filename}'),
         
         # [작업 3] Nginx 설정 파일 문법 검사 (아파치의 httpd -t 와 같은 역할)
         ("Nginx 문법 검사", "nginx -t"),
         
         # [작업 4] 문법에 이상이 없다면 Nginx 설정을 실시간으로 반영 (서비스 중단 없음)
         ("Nginx 설정 리로드", "nginx -s reload")
    ])

In [9]:
# from pprint import pprint

# 외부 함수들이 정의되어 있다고 가정합니다.

while True:
    print("==========메뉴=============")
    print("1.서버  2.네트워크   0.종료")
    print("==========================")
    
    choice1 = input("작업 선택: ")
    
    if choice1 == "1": # 서버
        print("1.Rocky  2.Ubuntu  0.종료")
        choice2 = input("서버 선택: ")
        
        if choice2 == "1":
            print("1.초기 설정 2. httpd 설치  0.종료")
            choice3 = input("작업 선택: ")
            if choice3 == "1":
                ip = input("아이피 입력: ")
                password = input("root 비밀번호 입력: ")
                init_rocky_package(ip=ip, user="root", pw=password, port=22)
            elif choice3 == "2":
                ip = input("아이피 입력: ")
                password = input("root 비밀번호 입력: ")
                is_user = input("유저가 이미 존재합니까?(아니라면 유저 생성됨, y/n): ") 
                username = input("유저명 입력: ")
                indexhtml_path = input("넣을 메인 페이지(index.html) 경로(없으면 n): ")
                user_password = input("유저 비밀번호 입력: ")
                
                # 💡 오타 수정: indexhmtl_path -> indexhtml_path
                if indexhtml_path == "n":
                    indexhtml_path = None
                    
                if is_user == "y":
                    is_user = True
                else:
                    is_user = False
                install_rocky_httpd(ip, password, user_name=username, new_user_passwd=user_password, is_user=is_user, index_html_path=indexhtml_path)
            else:
                print("서버 메뉴를 종료하고 메인으로 돌아갑니다.")
                # break를 걸지 않으면 하위 메뉴 구조상 바로 메인 메뉴로 튕겨 올라갑니다 (기존 유지)
                
        elif choice2 == "2":
            print("1.초기 설정  2. nginx 설치  0.종료")
            choice3 = input("작업 선택: ")
            if choice3 == "1":
                ip = input("아이피 입력: ")
                init_ubuntu_package(ip, username="root", password="asd123!@", port=22)
            elif choice3 == "2":
                ip = input("아이피 입력: ")
                password = input("root 비밀번호 입력: ")
                is_user = input("유저가 이미 존재합니까?(아니라면 유저 생성됨, y/n): ") 
                username = input("유저명 입력: ")
                indexhtml_path = input("넣을 메인 페이지(index.html) 경로(없으면 n): ")
                user_password = input("유저 비밀번호 입력: ")
                
                # 💡 오타 수정: indexhmtl_path -> indexhtml_path
                if indexhtml_path == "n":
                    indexhtml_path = None
                    
                if is_user == "y":
                    is_user = True
                else:
                    is_user = False
                install_ubuntu_nginx(ip, password, user_name=username, new_user_passwd=user_password, is_user=is_user, index_html_path=indexhtml_path)
            else:
                print("서버 메뉴를 종료하고 메인으로 돌아갑니다.")
                
    elif choice1 == "2": # 네트워크
        print("1. 라우터  2. 스위치")
        choice2 = input("작업 선택: ")
        
        if choice2 == "1":
            ip = input("아이피 입력: ")
            # with 구문의 변수를 con으로 통일하여 하위 로직과 맞춤
            with get_con(ip=ip, username="root", password="cisco", port=22, device_type="cisco_ios") as con:
                rs = get_device_role(con)
                rn = get_host_name(con)
                while True:
                    print("1. 인터페이스 정보 확인  2. 라우팅 테이블  3. DHCP  4. NAT  0. 종료")
                    work = input(f"{rs} {rn}작업 선택: ")
                    
                    if work == "1":
                        pprint(get_interface_brief(con))
                    elif work == "2":
                        print("1. 라우팅 정보 추가  2. 라우팅 테이블 확인  3. 라우팅 테이블 백업  4. 라우팅 테이블 복구  0. 종료")
                        sub_work = input(f"{rs} {rn}작업 선택: ")
                        if sub_work == "1":
                            nid = input("NID: ")
                            netmask = input("Netmask: ")
                            gateway = input("Gateway: ")
                            pprint(add_route_info(con, nid, netmask, gateway))
                        elif sub_work == "2":
                            pprint(get_routing_table(con))
                        elif sub_work == "3":
                            routing_table_json_path = input("json 경로: ")
                            pprint(store_routing_table(con, routing_table_json_path))
                        elif sub_work == "4":
                            routing_table_json_path = input("json 경로: ")
                            pprint(restore_routing_table(con, routing_table_json_path))
                        else:
                            break
                            
                    elif work == "3":
                        print("1. DHCP 서버 정보 확인  2. DHCP 서버 설정  3. DHCP 서버 백업  4. DHCP 서버 복구  0. 종료")
                        sub_work = input(f"{rs} {rn}작업 선택: ")
                        if sub_work == "1":
                            pprint(get_dhcp_info(con))
                        elif sub_work == "2":
                            pool_name = input("Pool Name: ")
                            nid = input("NID: ")
                            subnet_mask = input("Netmask: ")
                            gateway_ip = input("Gateway: ")
                            exlude_start = input("Exclude IP Start: ")
                            exclude_end = input("Exclude IP End: ")
                            dns_server = input("DNS: ")
                            pprint(router_dhcp_config(con, pool_name, subnet_mask, gateway_ip, exlude_start, exclude_end, dns_server))
                        elif sub_work == "3":
                            dhcp_setting_json_path = input("json 경로: ")
                            pprint(store_dhcp_setting(con, dhcp_setting_json_path))
                        elif sub_work == "4":
                            dhcp_setting_json_path = input("json 경로: ")
                            pprint(restore_dhcp_setting(con, dhcp_setting_json_path))
                        else:
                            break
                            
                    elif work == "4":
                        print("1. NAT 정보 확인  2. NAT 설정  3. NAT 백업  4. NAT 복구  0. 종료")
                        sub_work = input(f"{rs} {rn}작업 선택: ")
                        if sub_work == "1":
                            pprint(get_nat_pat_config(con))
                        elif sub_work == "2":
                            in_int = input("NAT Inside Interface: ")
                            out_int = input("NAT Outside Interface: ")
                            ip_netmask = input("IP/Netmask: ")
                            pprint(router_pat_nat_config(con, in_int, out_int, ip_netmask))
                        elif sub_work == "3":
                            nat_setting_json_path = input("json 경로: ")
                            pprint(store_nat_pat_setting(con, nat_setting_json_path))
                        elif sub_work == "4":
                            nat_setting_json_path = input("json 경로")
                            pprint(restore_nat_pat_setting(con, nat_setting_json_path))
                        else:
                            break
                    elif work == "0":
                        break
                    else:
                        break
                        
        elif choice2 == "2": # 스위치
            ip = input("아이피 입력: ")
            with get_con(ip=ip, username="root", password="cisco", port=22, device_type="cisco_ios") as con:
                rs = get_device_role(con)
                sn = get_host_name(con) # 스위치는 sn으로 받음
                while True:
                    print("1. 인터페이스 정보 확인  2. DHCP  0. 종료")
                    work = input(f"{rs} {sn}작업 선택: ") # rn을 sn으로 수정
                    if work == "1":
                        pprint(get_interface_brief(con))
                    elif work == "2":
                        print("1. DHCP 서버 정보 확인  2. DHCP 서버 설정  3. DHCP 서버 백업  4. DHCP 서버 복구  0. 종료")
                        sub_work = input(f"{rs} {sn}작업 선택: ")
                        if sub_work == "1":
                            pprint(get_dhcp_info(con))
                        elif sub_work == "2":
                            pool_name = input("Pool Name: ")
                            nid = input("NID: ")
                            subnet_mask = input("Netmask: ")
                            gateway_ip = input("Gateway: ")
                            exclude_start = input("Exclude IP Start: ")
                            exclude_end = input("Exclude IP End: ")
                            dns_server = input("DNS: ")
                            pprint(switch_dhcp_config(con, pool_name, nid, subnet_mask, gateway_ip, exclude_start, exclude_end, dns_server))
                        elif sub_work == "3":
                            dhcp_setting_json_path = input("json 경로: ")
                            pprint(store_dhcp_setting(con, dhcp_setting_json_path))
                        elif sub_work == "4":
                            dhcp_setting_json_path = input("json 경로: ")
                            pprint(restore_dhcp_setting(con, dhcp_setting_json_path))
                        else:
                            break
                    elif work == "0":
                        break
                    else:
                        break
                        
    elif choice1 == "0":
        break
    else:
        print("종료합니다")
        break

==========메뉴=============
1.서버  2.네트워크   0.종료


작업 선택:  0


In [34]:
install_ubuntu_nginx("172.16.7.4", "asd123!@", "aaaaaaaaaaa", "asd123!@", False, "./idx.html")


root@172.16.7.4:22에 SSH 접속 시도 중..


NoValidConnectionsError: [Errno None] Unable to connect to port 22 on 172.16.7.4

In [35]:
add_virtual_host_to_nginx("172.16.1.255", "asd123!@", "172.16.1.255", "81", "/home/hyogun/testfolder")


root@172.16.1.255:22에 SSH 접속 시도 중..

[디렉터리 만들기(없다면)] 시작...

[디렉터리 권한 777] 시작...

[index.html 생성] 시작...

[Nginx 블록 세팅] 시작...

[Nginx 문법 검사] 시작...

[시스템 메시지]:
2026/05/20 06:58:56 [warn] 1577#1577: conflicting server name "_" on 0.0.0.0:81, ignored
nginx: the configuration file /etc/nginx/nginx.conf syntax is ok
nginx: configuration file /etc/nginx/nginx.conf test is successful


[Nginx 설정 리로드] 시작...

[시스템 메시지]:
2026/05/20 06:58:56 [warn] 1578#1578: conflicting server name "_" on 0.0.0.0:81, ignored
2026/05/20 06:58:56 [notice] 1578#1578: signal process started



In [24]:
def install_PHP_lib_module(ip: str, root_password: str):
    maria_db_pasword="asd123!@"
    COMMANDS =[
          ("패키지 업데이트", f"apt update && sudo apt upgrade -y"),
          ("PHP PPA 저장소 추가", f"sudo add-apt-repository ppa:ondrej/php -y"),
          ("PHP 8.4 및 기본 모듈 설치", f"apt install -y php8.4 php8.4-cli php8.4-common php8.4-fpm"),
          ("시스템 기본 PHP 버전을 8.4로 활성화", f"sudo update-alternatives --set php /usr/bin/php8.4"),
          ("컴파일러 및 개발 필수 도구 설치", f"apt install -y build-essential gcc g++ libgdbm-dev libjpeg-dev libpng-dev libfreetype-dev libgd-dev zlib1g-dev libzip-dev"),
          ("PHP 8.4 핵심 기능 및 확장 모듈 설치", f"apt install -y php8.4 php8.4-fpm php8.4-cli php8.4-common php8.4-dev php-pear"),
          ("웹 개발 및 연동에 필요한 필수 PHP 모듈들 설치", f"apt install -y php8.4-bcmath php8.4-mbstring php8.4-odbc php8.4-gd php8.4-mysql php8.4-xml php8.4-zip php8.4-opcache php8.4-intl php8.4-pdo"),
          ("마리아DB 설치", f"apt install -y mariadb-server"),
          ("서비스 활성화 및 즉시 시작 ", f"systemctl enable --now mariadb"),
          ("마리아DB 설정파일 편집", f"sudo sed -i '7i default-character-set = utf8mb4' /etc/mysql/mariadb.conf.d/50-client.cnf"),
          ("마리아DB 재시작", f"systemctl restart mariadb"),
          ("마리아 DB 최소한의 보안설정", f"mysql -e \"ALTER USER 'root'@'localhost' IDENTIFIED BY '{maria_db_pasword}'; DELETE FROM mysql.user WHERE User=''; DROP DATABASE IF EXISTS test; FLUSH PRIVILEGES;\""),
          
     ]
    
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)



def install_PHP_myadmin(ip: str, ssh_root_password: str, db_root_password: str, web_root: str = "/home/testuser"):
    pma_app_password = 'asd123!@'

    # dbconfig 에러 차단용 사전 세팅
    debconf_cmd = (
        "echo 'phpmyadmin phpmyadmin/reconfigure-webserver multiselect ' | debconf-set-selections && "
        "echo 'phpmyadmin phpmyadmin/dbconfig-install boolean false' | debconf-set-selections"
    )

    # 💡 인계받은 web_root 경로가 Nginx 설정에 동적으로 주입되도록 f-string 문법을 정교화했습니다.
    # (파이썬 중괄호 충돌 방지를 위해 {{ }} 로 이스케이프 처리 완료)
    nginx_config = f"""cat << 'EOF' > /etc/nginx/sites-available/default
server {{
    listen 80 default_server;
    listen [::]:80 default_server;

    root {web_root};
    index index.php index.html index.htm;

    server_name _;

    location / {{
        try_files $uri $uri/ =404;
    }}

    location ~ \\.php$ {{
        include snippets/fastcgi-php.conf;
        fastcgi_pass unix:/run/php/php8.4-fpm.sock;
    }}
}}
EOF"""

    COMMANDS = [
        ("패키지 목록 업데이트", "DEBIAN_FRONTEND=noninteractive apt-get update"),
        ("debconf-utils 설치", "DEBIAN_FRONTEND=noninteractive apt-get install -y debconf-utils"),
        ("phpMyAdmin 사전 설정 주입", debconf_cmd),
        ("phpMyAdmin 무인 설치", "DEBIAN_FRONTEND=noninteractive apt-get install -y phpmyadmin"),
        
        # 안전한 수동 DB 세팅 프로세스
        ("phpMyAdmin 전용 DB 생성", f"mysql -u root -p'{db_root_password}' -e 'CREATE DATABASE IF NOT EXISTS phpmyadmin;'"),
        ("phpMyAdmin 전용 유저 생성", f"mysql -u root -p'{db_root_password}' -e \"CREATE USER IF NOT EXISTS 'phpmyadmin'@'localhost' IDENTIFIED BY '{pma_app_password}';\""),
        ("phpMyAdmin 권한 부여", f"mysql -u root -p'{db_root_password}' -e \"GRANT ALL PRIVILEGES ON phpmyadmin.* TO 'phpmyadmin'@'localhost'; FLUSH PRIVILEGES;\""),
        ("phpMyAdmin 기본 스키마 주입", f"mysql -u root -p'{db_root_password}' phpmyadmin < /usr/share/phpmyadmin/sql/create_tables.sql"),
        
        # 환경설정 보완 및 Nginx 연동
        ("php cookie->http 변경", "sed -i \"s/auth_type'] = 'cookie'/auth_type'] = 'http'/g\" /etc/phpmyadmin/config.inc.php"),
        ("PHP mbstring 모듈 활성화", "phpenmod mbstring"),
        
        # 💡 진짜 웹 루트 폴더 하위에 정확하게 바로가기가 생성되도록 수정되었습니다.
        ("Nginx 연동 심볼릭 링크 생성", f"ln -sf /usr/share/phpmyadmin {web_root}/phpmyadmin"),
        ("Nginx PHP 가동 설정 주입", nginx_config),
        ("Nginx 설정 활성화 링크 생성", "ln -sf /etc/nginx/sites-available/default /etc/nginx/sites-enabled/default"),
        
        # 권한 제어 및 홈 디렉토리 개방
        ("권한이동", f"chown -R www-data:www-data /usr/share/phpmyadmin {web_root}"),
        ("phpMyAdmin 권한부여", "chmod -R 755 /usr/share/phpmyadmin"),
        ("홈 디렉토리 권한 개방", f"chmod 755 {web_root}"),
        
        # 최종 서비스 갱신
        ("PHP-FPM 재시작", "systemctl restart php8.4-fpm"),
        ("nginx 웹 서버 재시작", "systemctl restart nginx")
    ]
    
    client = get_ssh_client(ip, ssh_root_password, "root", 22)
    try:
        run_cmd(client, COMMANDS)
    finally:
        client.close()  

In [25]:
init_ubuntu_package("172.16.1.5", username="root", password="asd123!@", port=22)


install_ubuntu_nginx("172.16.1.5", "asd123!@", "testuser", "asd123!@", False, None)

install_PHP_lib_module("172.16.1.5", "asd123!@")

install_PHP_myadmin("172.16.1.5", "asd123!@", "asd123!@")

[172.16.1.5] 서버에 SSH 연결을 시도합니다...
연결 성공! 자동 설정 스크립트를 원격으로 실행합니다. (몇 분 정도 소요됩니다)

Hit:1 http://kr.archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://kr.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 http://kr.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 http://kr.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2,056 kB]
Get:6 http://kr.archive.ubuntu.com/ubuntu noble-updates/main Translation-en [362 kB]
Get:7 http://kr.archive.ubuntu.com/ubuntu noble-updates/main amd64 Components [177 kB]
Get:8 http://kr.archive.ubuntu.com/ubuntu noble-updates/main amd64 c-n-f Metadata [17.4 kB]
Get:9 http://kr.archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [3,281 kB]
Get:10 http://kr.archive.ubuntu.com/ubuntu noble-updates/restricted Translation-en [761 kB]
Get:11 http://kr.archive.ubuntu.com/ubuntu noble-updates/restricted amd64 c-n-f Metadata [456 B]
Ge

In [ ]:
def install_vsftpd(ip: str, root_password: str):
   
    COMMANDS =[
         ("업데이트", "dnf -y upgrade && dnf update -y"),
         ("vsftpd 설","dnf install -y vsftpd ftp"),
         ("부팅시 vsftpd 자동실행", "systemctl enable --now vsftpd"),
         ("vsftpd PAM 쉘 검증 주석 처리", "sudo sed -i 's/^auth.*required.*pam_shells.so/#&/' /etc/pam.d/vsftpd"),
         ("vsftpd 서비스 재시작", "sudo systemctl restart vsftpd")
]
     ]
    
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)

In [40]:
def set_user_nologin(ip: str, root_password: str, target_user: str):
    
    # 1. /etc/passwd에서 유저를 찾고(grep), 존재하면 쉘을 바꾸고(usermod), 성공하면 재시작(systemctl)
    # 기호 '^'와 ':'를 써서 정확히 일치하는 계정명만 찾도록 안전장치를 둡니다.
    integrated_cmd = (
        f"grep -q '^{target_user}:' /etc/passwd && "
        f"sudo usermod -s /sbin/nologin {target_user} && "
        f"sudo systemctl restart vsftpd"
    )
    
    COMMANDS = [
        (f"[{target_user}] 계정 검증 및 sbin/nologin 설정 일괄 처리", integrated_cmd)
    ]

    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)


#set_user_nologin_single_cmd("172.16.10.10", "asd123!@", "test")


root@172.16.10.10:22에 SSH 접속 시도 중..

[[test] 계정 검증 및 sbin/nologin 설정 일괄 처리] 시작...


In [1]:
##Zone

# zone 파일 생성 함
from datetime import datetime

def create_domain_zone_file(ip: str, root_password: str, domain: str, target_ip: str):
  
    zone_file_path = f"/var/named/{domain}.zone"
    
    # 오늘 날짜를 YYYYMMDD 형태로 포맷팅하고 뒤에 01 번호를 붙임 
    today_serial = datetime.now().strftime("%Y%m%d") + "01"
    
    # zone 파일 기본 뼈대 내용
    zone_content = (
        f"$TTL 1D\\n"
        f"@\\tIN SOA  ns.{domain}. root.{domain}. (\\n"
        f"                {today_serial}   ; serial\\n"
        f"                1D      ; refresh\\n"
        f"                1H      ; retry\\n"
        f"                1W      ; expire\\n"
        f"                3H )    ; minimum\\n"
        f"    NS    ns.{domain}.\\n"
        f"ns\\tIN\\tA\\t{target_ip}\\n"
        f"@\\tIN\\tA\\t{target_ip}\\n"
    )
    
    # 파일이 없을 때만 초기 생성 진행 (기존 파일 덮어쓰기 방지)
    integrated_cmd = (
        f"[ ! -f {zone_file_path} ] && "
        f"echo -e '{zone_content}' | sudo tee {zone_file_path} && "
        f"sudo chown root:named {zone_file_path} && "
        f"sudo chmod 640 {zone_file_path}"
    )
    
    COMMANDS = [
        (f"[{domain}] 날짜형 Serial 반영 zone 파일 최초 생성", integrated_cmd)
    ]
    
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)

#zone 파일에 record 만 추
def add_named_zone_record(ip: str, root_password: str, domain: str, host: str,  value: str, record_type: str="A"):
  
    zone_file_path = f"/var/named/{domain}.zone"
    new_record = f"{host}\\tIN\\t{record_type}\\t{value}"
    
    # 1. 파일이 존재하는지 확인
    # 2. 동일한 레코드가 이미 있는지 중복 체크
    # 3. 파일 맨 아래에 새 레코드 한 줄 추가 (개행 포함)
    # 4. 기존 파일에서 '; serial' 주석이 달린 줄의 앞 숫자를 추출해 +1 증가시킨 후 파일 갱신 (sed 사용)
    # 5. named 서비스 재시작
    integrated_cmd = (
        f"[ -f {zone_file_path} ] && "
        f"! sudo grep -q '{new_record}' {zone_file_path} && "
        f"echo -e '{new_record}' | sudo tee -a {zone_file_path} && "
        f"current_serial=$(sudo grep -E ';\\\\s*serial' {zone_file_path} | awk '{{print $1}}') && "
        f"next_serial=$((current_serial + 1)) && "
        f"sudo sed -i \"s/$current_serial.*;\\\\s*serial/$next_serial\\t\\t; serial/g\" {zone_file_path} && "
        f"sudo systemctl restart named"
    )
    
    COMMANDS = [
        (f"[{domain}] 새 레코드 '{host}' 추가 및 Serial 업데이트", integrated_cmd)
    ]
    
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)


#/etc/named.rfc1912.zones에 zone 등록
def update_rfc_config(ip: str, root_password: str, domain: str):

    target_conf = "/etc/named.rfc1912.zones"
    zone_file_name = f"{domain}.zone"
    
    zone_config = (
        f'\\nzone "{domain}" IN {{\\n'
        f'\\ttype master;\\n'
        f'\\tfile "{domain}.zone";\\n'
        f'\\tallow-update {{ none; }};\\n'
        f'}};\\n'
    )
    
    # 중복 추가 방지 처리 후 파일 하단에 추가하고 서비스 재시작
    integrated_cmd = (
        f"sudo grep -q 'zone \"{domain}\"' {target_conf} || "
        f"(echo -e '{zone_config}' | sudo tee -a {target_conf} && "
        f"sudo systemctl restart named)"
    )
    
    COMMANDS = [
        (f"[{domain}] RFC 설정 파일에 zone 블록 추가 및 named 재시작", integrated_cmd)
    ]
    
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)


#servername 포함하여 httpd virtual host 생성
def add_virtual_host_to_httpd_with_servername(ip: str, root_password: str, record: str, domain: str, vhostfolderpath: str, vhostip: str="*", vhostport: str="80"):
    
    vhost_config = (
        fr"\n<VirtualHost {vhostip}:{vhostport}>\n"
        fr'    ServerName "{record}.{domain}"\n'       
        fr'    DocumentRoot \"{vhostfolderpath}\"\n'
        fr'    <Directory \"{vhostfolderpath}\">\n'
        fr"        Require all granted\n"
        fr"    </Directory>\n"
        fr"</VirtualHost>"
    )

    run_cmd(get_ssh_client(ip, root_password, "root", 22), [
         # 이미 폴더가 있으면 [시스템 메시지]에 File exists가 뜨지만 아래 세팅으로 정상 진행됩니다.
         ("디렉터리 만들기(없다면)", f"mkdir {vhostfolderpath}"),
         
         # 이 한 줄 명령어가 vhost.conf 맨 뒤에 이쁘게 내용을 추가해 줍니다.
         ("vhost.conf 세팅", f'echo -e "{vhost_config}" >> /etc/httpd/conf.d/vhost.conf')
    ])



#위 함수들 통합 - 새로운 zone 생성 및 httpd설정 
def add_new_domain(ip: str, root_password: str, domain: str, record: str, domain_ip: str, vhostfolderpath:str):
    create_domain_zone_file(ip, root_password, domain, domain_ip)
    add_named_zone_record(ip, root_password, domain, record, domain_ip)
    update_rfc_config(ip, root_password, domain)
    add_virtual_host_to_httpd_with_servername(ip, root_password, record, domain, vhostfolderpath)

    cmd = (
        "sudo systemctl restart named && "
        "sudo systemctl restart httpd"
    )
    
    COMMANDS = [
        ("httpd, named 재시작", cmd)
    ]
    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)
    

In [27]:
rocky_setup_mariadb("172.16.1.4", "asd123!@", "asd123!@")


root@172.16.1.4:22에 SSH 접속 시도 중..

[dnf 업데이트] 시작...
  마지막 메타자료 만료확인(0:01:23 이전): 2026년 06월 04일 (목) 오후 02시 55분 20초.
  종속성이 해결되었습니다.
  꾸러미                      구조   버전                        저장소     크기
  설치 중:
  kernel                      x86_64 5.14.0-687.12.1.el9_8       baseos    934 k
  향상 중:
  NetworkManager              x86_64 1:1.54.3-2.el9              baseos    2.4 M
  NetworkManager-libnm        x86_64 1:1.54.3-2.el9              baseos    1.9 M
  NetworkManager-team         x86_64 1:1.54.3-2.el9              baseos     37 k
  NetworkManager-tui          x86_64 1:1.54.3-2.el9              baseos    248 k
  audit                       x86_64 3.1.5-8.el9                 baseos    253 k
  audit-libs                  x86_64 3.1.5-8.el9                 baseos    121 k
  binutils                    x86_64 2.35.2-72.el9               baseos    4.5 M
  binutils-gold               x86_64 2.35.2-72.el9               baseos    734 k
  bzip2-libs                  x86_64 1.0.8-11.el9     